In [1]:
import batting_statlines as stat 
import pandas as pd 
import numpy as np 
import mysql.connector as sql
from IPython import display
from collections import namedtuple

pd.options.display.max_columns = None

In [17]:
fangraphs_board = pd.read_csv('pitching_1947_2001.csv')
fangraphs_board.rename(columns={'season': 'yearID', 'team': 'teamID'}, inplace = True)
fangraphs_board.drop(['BABIP', 'G', 'GS', 'HR/FB', 'xFIP', 'xFIP-', 'playerid', 'K/9', 'BB/9', 'K/9+', 'BB/9+'], axis=1, inplace=True)
fangraphs_board = stat.order_by(fangraphs_board, ['yearID', 'teamID'], True)
fangraphs_board = fangraphs_board[fangraphs_board.teamID != '- - -']
for col in ['K%', 'BB%']:
    fangraphs_board[col] = fangraphs_board[col].replace('%', '', regex=True)
    fangraphs_board[col] = pd.to_numeric(fangraphs_board[col])

## Here We Go!

The first thing that I want to do, is to group everybody into teams. In order to do this analysis, I'm going to be looking at Johnson and Schilling's numbers together as a single aggregate. I think that would be the best, becuase I'm going to compare him to other top 2 pitchers on teams so we're really comparing "groups" of players rather than individual players.

### The Plan

What I plan on doing is first, create the aggregate values for each column, for the top two pitchers in a team each year. After I do that I want to rank each pair of staff aces for each statistic. From there, create an average score for how they rank in each statistic to find the most "dominant" pairing of pitchers. Then, look across each year and see who has the highest "dominance" score.

In [18]:
years = fangraphs_board.yearID.unique()
teams = fangraphs_board.teamID.unique()
dont_do = ['name', 'teamID']
team_dict = namedtuple('team_avgs', 'col_name dict')
def make_avgs_dict(year, team):
    team_dict = {}
    team_df = fangraphs_board[(fangraphs_board.yearID == year) & (fangraphs_board.teamID == team)]
    team_df = stat.order_by(team_df, 'WAR', False)
    team_df = team_df.head(2)
    for col in team_df.columns:
        if col not in dont_do:
            team_dict[col] = team_df[col].mean()
    return team_dict

dicts = {}
for year in years:
    for team in teams:
        years_teams = fangraphs_board[fangraphs_board.yearID == year].teamID.unique()
        if team in years_teams:
            t = team_dict(col_name=f'{team}_{year}', dict=make_avgs_dict(year, team))
            dicts[t.col_name] = t.dict


In [48]:
top2_aggs = pd.DataFrame(data=dicts.values(), index=dicts.keys())
top2_aggs.reset_index(inplace=True)
top2_aggs['index'] = top2_aggs['index'].apply(lambda x: x.split('_')[0])
top2_aggs.rename(columns={'index': 'teamID'}, inplace=True)

In [49]:
year_2001 = top2_aggs[top2_aggs.yearID == 2001]
year_2001.reset_index(drop=True, inplace=True)
counting_cols = ['W', 'IP', 'K%', 'K/BB+', 'WAR', 'K%+']
def rank_it(df, counting):
    ranked_df = df.copy()
    for col in ranked_df.columns:
        if col == 'teamID' or col == 'yearID':
            continue
        elif col in counting:
            ranked_df[col] = df[col].rank(method='first', ascending=False)
        else:
            ranked_df[col] = df[col].rank(method='first')
    ranked_df['composite'] = ranked_df.filter(regex = '[^yearID]').mean(axis=1)
    return ranked_df
rank_2001 = rank_it(year_2001, counting_cols)
rank_2001 = stat.order_by(rank_2001, 'composite', True)

In [59]:
ranked_years = []
for year in top2_aggs.yearID.unique():
    year_df = top2_aggs[top2_aggs.yearID == year]
    ranked_df = rank_it(year_df, counting_cols)
    ranked_years.append(ranked_df)
ranked_aggs = pd.concat(ranked_years)
ranked_aggs_ordered = stat.order_by(ranked_aggs, 'composite', True)

In [63]:
top20 = ranked_aggs_ordered.head(20)
top20_appearances = {}
for team in top20.teamID:
    if not team in top20_appearances.keys():
        top20_appearances[team] = 1
    else:
        top20_appearances[team] += 1
top20['Appearances'] = top20.teamID.apply(lambda x: top20_appearances[x])
top20[top20.Appearances > 1].teamID.nunique()

7

## The Results

Wow! There definitely are a lot of teams who were dominant over the years, but still TWO top 20 performances among over 1786 other teams is just ridiculous. Only 7 of the 30 teams in the sample have had multiple teams in the top 20. I think what I want to do next is look at groups of pairings. So, Schilling and Johnson would have had about a 2.5ish average composite score in those two top 20 years. I wonder what the score would be for their entire time together on the diamondbacks? What are some other electric duos that stuck together for a while. Time to find out!